# 扩展阅读：MTP 投机解码——从"偷跑一个"到"偷跑一串"

在 [01-3](./01-3-token_generates_token.ipynb) 中，我们介绍了 MTP 模块的基本结构。这个扩展阅读将深入探讨 MTP 如何真正加速推理——**投机解码 (Speculative Decoding)**。

投机解码有两种常见策略：
- **K=1（单草稿）**：每次偷跑 1 个 token，下一轮验证。这是最简单、最稳定的实现，很多实际部署（包括最初的 MTP 论文）都采用这种方式，加速比上限 **2x**。
- **K>1（多草稿）**：一次偷跑 K 个 token，一次验证 K 个。这是投机解码的**完整形态**，加速比上限 **K+1**，但对草稿质量要求更高。

本节我们将：
1. 回顾 MTP 模块的结构与前向传播
2. 理解 K=1 的单草稿投机解码（加速比上限 2x）
3. 实现多草稿并行投机解码（加速比上限 K+1）
4. 对比两种方式的加速效果，并分析多草稿策略的局限性

In [1]:
from modelscope import AutoTokenizer, AutoModelForCausalLM
import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors import safe_open
from transformers.models.qwen3_5.modeling_qwen3_5 import Qwen3_5DecoderLayer, Qwen3_5RMSNorm
from transformers.masking_utils import create_causal_mask
import glob, os, copy

tokenizer_id = "Qwen/Qwen3.5-0.8B"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_id)

device = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
)
if os.environ.get('FORCE_CPU'):
    device = torch.device('cpu')
model = AutoModelForCausalLM.from_pretrained(tokenizer_id).to(device).eval()
dtype = next(model.parameters()).dtype
print(f"模型加载完成: {tokenizer_id} (device: {device})")

cache_dir = os.path.expanduser('~/.cache/modelscope/hub/models/Qwen/Qwen3.5-0.8B')
if not os.path.exists(cache_dir):
    cache_dir = os.path.expanduser('~/.cache/modelscope/hub/models/Qwen/Qwen3___5-0___8B')
sf_files = glob.glob(os.path.join(cache_dir, '*.safetensors'))
mtp_state = {}
for f in sf_files:
    with safe_open(f, framework='pt') as sf:
        for key in sf.keys():
            if key.startswith('mtp.'):
                mtp_state[key[4:]] = sf.get_tensor(key)

config = model.config
mtp_config = copy.deepcopy(config)
mtp_config.layer_types = ["full_attention"]

mtp_layer = Qwen3_5DecoderLayer(mtp_config, layer_idx=0).to(dtype=dtype, device=device)
layer_state = {}
for key, val in mtp_state.items():
    if key.startswith('layers.0.'):
        layer_state[key[len('layers.0.'):]] = val.to(dtype=dtype, device=device)
mtp_layer.load_state_dict(layer_state, strict=False)
mtp_layer.eval()

mtp_fc_w = mtp_state['fc.weight'].to(dtype=dtype, device=device)
mtp_pre_fc_norm_e = Qwen3_5RMSNorm(config.hidden_size, eps=config.rms_norm_eps).to(dtype=dtype, device=device)
mtp_pre_fc_norm_e.weight = nn.Parameter(mtp_state['pre_fc_norm_embedding.weight'].to(dtype=dtype, device=device))
mtp_pre_fc_norm_h = Qwen3_5RMSNorm(config.hidden_size, eps=config.rms_norm_eps).to(dtype=dtype, device=device)
mtp_pre_fc_norm_h.weight = nn.Parameter(mtp_state['pre_fc_norm_hidden.weight'].to(dtype=dtype, device=device))
mtp_output_norm = Qwen3_5RMSNorm(config.hidden_size, eps=config.rms_norm_eps).to(dtype=dtype, device=device)
mtp_output_norm.weight = nn.Parameter(mtp_state['norm.weight'].to(dtype=dtype, device=device))

mtp_params = sum(v.numel() for v in mtp_state.values())
main_params = sum(p.numel() for p in model.model.parameters())
print(f"MTP 参数量: {mtp_params:,} (仅占主模型 {main_params:,} 的 {mtp_params/main_params:.2%})")
print("MTP 推理组件构建完成")

2026-05-22 05:24:20,592 - modelscope - INFO - Target directory already exists, skipping creation.


2026-05-22 05:24:21,884 - modelscope - INFO - Target directory already exists, skipping creation.
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

模型加载完成: Qwen/Qwen3.5-0.8B (device: mps)
MTP 参数量: 20,452,864 (仅占主模型 752,393,024 的 2.72%)
MTP 推理组件构建完成


## 1. MTP 前向传播回顾

MTP 模块的核心逻辑：对序列中每个位置 i，拼接 `hidden_states[i]` 和 `embed_tokens[i+1]`（下一个 token 的 embedding），经过 fc 投影 → 1 层 Transformer → RMSNorm → lm_head，预测 token+2。

```
位置 i 的 MTP 输入 = fc(RMSNorm(embed_tokens[i+1]) ⊕ RMSNorm(hidden_states[i]))
                                          ↑ "下一个是什么"        ↑ "上下文理解"
```

关键：MTP 之所以能预测 token+2，是因为它**同时看到了**主模型的上下文理解（hidden_states）和下一个 token 的嵌入（embedding）——相当于"已知 token+1，预测 token+2"。

In [2]:
def mtp_forward_once(cur_ids, cur_hidden, next_token_id):
    """MTP 单次前向传播：基于当前 hidden_states 和下一个 token，预测 token+2"""
    with torch.no_grad():
        input_embeds = model.model.embed_tokens(cur_ids)
        next_embed = model.model.embed_tokens(torch.tensor([[next_token_id]], device=device))
        next_embeds = torch.cat([input_embeds[:, 1:, :], next_embed], dim=1)
        normed_e = mtp_pre_fc_norm_e(next_embeds)
        normed_h = mtp_pre_fc_norm_h(cur_hidden)
        combined = torch.cat([normed_e, normed_h], dim=-1)
        projected = F.linear(combined, mtp_fc_w)
        seq_len = projected.shape[1]
        # MRoPE 需要 3 个轴的位置 ID: (3, batch_size, seq_len)
        # 3 = Temporal + Height + Width，文本输入时三个轴使用相同位置
        n_mrope_axes = 3
        position_ids = torch.arange(seq_len, device=device).view(1, 1, -1).expand(n_mrope_axes, 1, -1)
        position_embeddings = model.model.rotary_emb(projected, position_ids)
        causal_mask = create_causal_mask(
            config=mtp_config, inputs_embeds=projected,
            attention_mask=None, past_key_values=None,
            position_ids=position_ids[0],
        )
        layer_out = mtp_layer(
            projected, position_embeddings=position_embeddings,
            attention_mask=causal_mask, position_ids=position_ids[0],
        )
        mtp_hidden = mtp_output_norm(layer_out)
        mtp_logits = model.lm_head(mtp_hidden[:, -1:, :])
        draft_id = torch.argmax(F.softmax(mtp_logits.float(), dim=-1), dim=-1).item()
        return draft_id, mtp_hidden[:, -1:, :]

input_text = "你爱我，我爱你，蜜雪"
input_ids = tokenizer.encode(input_text, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.model(input_ids)
    hidden_states = outputs.last_hidden_state
    main_logits = model.lm_head(hidden_states[:, -1:, :])
    next_id = torch.argmax(F.softmax(main_logits.float(), dim=-1), dim=-1).item()

draft_id, _ = mtp_forward_once(input_ids, hidden_states, next_id)
print(f"输入: {input_text}")
print(f"主模型预测 token+1: {tokenizer.decode([next_id])}")
print(f"MTP 预测 token+2: {tokenizer.decode([draft_id])}")

输入: 你爱我，我爱你，蜜雪
主模型预测 token+1: 冰
MTP 预测 token+2: 城


## 2. K=1 单草稿投机解码：每轮验证1个草稿

最简单的投机解码是 K=1：MTP 每次偷跑1个草稿 token，下一轮主模型前向时**顺便**验证它。这是投机解码最基本、最稳定的实现形式，很多实际部署都采用这种方式。

关键：每轮只有**1次主模型前向**，同时完成两件事：
1. 验证上一轮的草稿 token
2. 预测下一个 token

```
轮次1: 主模型前向 → 预测 token+1, MTP偷跑草稿 token+2
轮次2: 主模型前向(输入含草稿) → 验证 token+2 + 预测 token+3, MTP偷跑草稿 token+4
轮次3: 主模型前向(输入含草稿) → 验证 token+4 + 预测 token+5, MTP偷跑草稿 token+6
```

如果草稿100%正确，每轮（除首轮）产出2个token，只需1次主模型前向 → **加速比上限 2x**。

这是 K=1 的单草稿投机解码，已经比传统自回归快了不少。但真正的飞跃在于增大 K——一次偷跑一串。

In [3]:
def speculative_decoding_k1(input_text, max_new_tokens=12, verbose=True):
    """K=1 单草稿投机解码：每轮1次主模型前向，验证1个草稿 + 预测下一个token"""
    input_ids = tokenizer.encode(input_text, return_tensors="pt").to(device)
    original_len = len(input_ids[0])
    confirmed = input_ids[0].tolist()
    draft_token = None
    total_main = 0
    total_mtp = 0

    if verbose:
        print(f"输入: {input_text}")
        print("=" * 70)

    while len(confirmed) - original_len < max_new_tokens:
        if draft_token is None:
            current_ids = torch.tensor([confirmed], device=device)
        else:
            current_ids = torch.tensor([confirmed + [draft_token]], device=device)

        with torch.no_grad():
            outputs = model.model(current_ids)
            hidden_states = outputs.last_hidden_state
        total_main += 1

        if draft_token is not None:
            verify_logits = model.lm_head(hidden_states[:, -2:-1, :])
            verified_id = torch.argmax(F.softmax(verify_logits.float(), dim=-1), dim=-1).item()
            if verified_id == draft_token:
                confirmed.append(draft_token)
                if verbose:
                    print(f"  草稿 '{tokenizer.decode([draft_token])}' 通过! +1 token")
            else:
                confirmed.append(verified_id)
                if verbose:
                    print(f"  草稿 '{tokenizer.decode([draft_token])}' 失败, 修正为 '{tokenizer.decode([verified_id])}'")
                draft_token = None

        last_logits = model.lm_head(hidden_states[:, -1:, :])
        next_id = torch.argmax(F.softmax(last_logits.float(), dim=-1), dim=-1).item()
        confirmed.append(next_id)

        if verbose:
            print(f"  主模型预测: '{tokenizer.decode([next_id])}'")

        if next_id == tokenizer.eos_token_id:
            break

        draft_id, _ = mtp_forward_once(current_ids, hidden_states, next_id)
        draft_token = draft_id
        total_mtp += 1

        if verbose:
            print(f"  MTP草稿: '{tokenizer.decode([draft_id])}'")
            print(f"  当前: {tokenizer.decode(confirmed)}\n")

    new_tokens = len(confirmed) - original_len
    final_text = tokenizer.decode(confirmed)

    if verbose:
        print(f"{'=' * 70}")
        print(f"最终结果: {final_text}")
        print(f"主模型前向: {total_main} 次, MTP前向: {total_mtp} 次")
        print(f"产出 Token: {new_tokens} 个, 加速比: {new_tokens/total_main:.2f}x")

    return final_text, total_main, total_mtp, new_tokens

result, main_fwd, mtp_fwd, n_tokens = speculative_decoding_k1("你爱我，我爱你，蜜雪")
print(f"\nK=1 单草稿投机解码加速比上限: 2x (每轮最多产2个token, 只需1次主模型前向)")

输入: 你爱我，我爱你，蜜雪
  主模型预测: '冰'
  MTP草稿: '城'
  当前: 你爱我，我爱你，蜜雪冰

  草稿 '城' 通过! +1 token
  主模型预测: '，'
  MTP草稿: '你'
  当前: 你爱我，我爱你，蜜雪冰城，

  草稿 '你' 通过! +1 token
  主模型预测: '爱我'
  MTP草稿: '，'
  当前: 你爱我，我爱你，蜜雪冰城，你爱我

  草稿 '，' 通过! +1 token
  主模型预测: '我爱你'
  MTP草稿: '，'
  当前: 你爱我，我爱你，蜜雪冰城，你爱我，我爱你

  草稿 '，' 通过! +1 token
  主模型预测: '蜜'
  MTP草稿: '雪'
  当前: 你爱我，我爱你，蜜雪冰城，你爱我，我爱你，蜜

  草稿 '雪' 通过! +1 token
  主模型预测: '冰'
  MTP草稿: '城'
  当前: 你爱我，我爱你，蜜雪冰城，你爱我，我爱你，蜜雪冰

  草稿 '城' 通过! +1 token
  主模型预测: '，'
  MTP草稿: '你'
  当前: 你爱我，我爱你，蜜雪冰城，你爱我，我爱你，蜜雪冰城，

最终结果: 你爱我，我爱你，蜜雪冰城，你爱我，我爱你，蜜雪冰城，
主模型前向: 7 次, MTP前向: 7 次
产出 Token: 13 个, 加速比: 1.86x

K=1 单草稿投机解码加速比上限: 2x (每轮最多产2个token, 只需1次主模型前向)


## 3. K个草稿的并行投机解码：一次偷跑一串

真正的投机解码不是每次只偷跑1个，而是**一次偷跑K个草稿，一次验证K个**。

### 核心流程（每轮只有1次主模型前向）

```
阶段一：草稿生成（用轻量的 MTP 模块，快速生成 K 个草稿）
  MTP前向1 → 草稿 token+2
  MTP前向2 → 草稿 token+3
  MTP前向3 → 草稿 token+4
  → K 次 MTP 前向，生成 K 个草稿

阶段二：一次主模型前向，同时完成验证 + 预测
  输入 = [已确认序列] + [草稿1, 草稿2, ..., 草稿K]
  ↓
  主模型前向 → 每个位置都有 logits
  ↓
  取位置 C-1 的 logits → 验证草稿1
  取位置 C   的 logits → 验证草稿2
  取位置 C+1 的 logits → 验证草稿3
  ...
  取最后一个位置的 logits → 预测下一个 token
  ↓
  找到第一个不匹配的位置 j：
    - 接受草稿1..j-1（全部正确）
    - 在位置 j 用主模型的 logits 修正
    - 丢弃草稿j+1..K（基于错误前提，全部作废）
```

### 为什么验证K个和验证1个的计算量几乎相同？

主模型的前向传播天然会为序列中的**每个位置**计算 hidden states 和 logits。把 K 个草稿 token 拼在输入后面，主模型照常计算整个序列——我们只是从输出中多提取了 K 个位置的 logits 做验证。**验证的边际成本极低**，这才是真正的加速来源。

### 加速比上限

如果 K 个草稿全部正确，1次主模型前向产出 K+1 个新 token（K 个草稿 + 末尾额外预测的1个），相比传统自回归的 1 token/次，**加速比上限为 K+1**。

### ⚠️ 单 MTP 头生成多草稿的严重局限

Qwen3.5-0.8B 只有 1 个 MTP 头，只能直接预测 token+2。要预测更远的 token，需要自回归地多次运行 MTP，每次将 MTP 自己的输出拼接为 hidden_states——**但这是一种不正确的近似**：

- 训练时，MTP 的输入是主模型的**真实** hidden_states
- 生成第2个及之后的草稿时，输入的是 MTP 自己的输出——主模型从未处理过这些 token，拼接出的 hidden states 是**虚构的**，与主模型的真实表示毫无关系
- 这不是"质量递减的合理近似"，而是**几乎等同于随机猜测**

这意味着：**单 MTP 头 + 自回归拼接，无法可靠地生成高质量的多草稿**。K=1（单草稿）才是这种架构下最稳妥的选择。

下面的代码仍然实现了多草稿流程，目的是让你**完整理解投机解码的概念**——但请记住，其中草稿生成部分存在上述缺陷。在实际部署中，多草稿投机解码需要更好的草稿来源（见本节末尾的讨论）。

In [4]:
def generate_k_drafts(input_ids, hidden_states, next_token_id, k=3):
    """用 MTP 模块自回归地生成 K 个草稿 token

    ⚠️ 重要说明：这个函数仅用于演示投机解码的概念流程，
    其中的草稿生成方式存在严重缺陷，不适合实际使用。

    问题在于：生成第2个及之后的草稿时，我们将 MTP 自己的输出
    拼接到 hidden_states 中——但主模型从未处理过这些新增 token，
    所以这些拼接出来的 hidden states 是"虚构的"，与主模型的
    真实 hidden states 毫无关系。在此基础上继续调用 MTP，
    其语义已严重偏离训练时的条件。

    实际部署中，多草稿生成通常采用以下方式之一：
    - 多头 MTP（如 DeepSeek-V3 的 4 个头），每个头独立依赖
      原始主模型 hidden_states，无需自回归拼接
    - 树状注意力（Tree Attention），一次性验证多条草稿路径
    - 独立的小型草稿模型，有自己完整的推理能力
    """
    draft_tokens = []
    cur_hidden = hidden_states
    cur_ids = input_ids
    cur_next_id = next_token_id

    for i in range(k):
        draft_id, mtp_last_hidden = mtp_forward_once(cur_ids, cur_hidden, cur_next_id)
        draft_tokens.append(draft_id)
        cur_hidden = torch.cat([cur_hidden, mtp_last_hidden], dim=1)
        cur_ids = torch.cat([cur_ids, torch.tensor([[cur_next_id]], device=device)], dim=1)
        cur_next_id = draft_id

    return draft_tokens

draft_tokens = generate_k_drafts(input_ids, hidden_states, next_id, k=3)
draft_texts = [tokenizer.decode([d]) for d in draft_tokens]
print(f"输入: {input_text}")
print(f"主模型预测 token+1: '{tokenizer.decode([next_id])}'")
print(f"MTP 生成 3 个草稿: {draft_texts}")
print()
print("⚠️ 注意：第1个草稿用了主模型的真实 hidden_states，质量较高")
print("但第2、3个草稿用了 MTP 自己的输出拼接——这些 hidden_states 是虚构的!")
print("主模型从未处理过这些 token，拼接出的 hidden states 没有依据，")
print("所以后续草稿几乎等同于随机猜测，而非合理近似。")
print("验证阶段会纠正所有错误，所以最终结果不受影响——")
print("但草稿质量低意味着多草稿策略可能无法带来加速。")

输入: 你爱我，我爱你，蜜雪
主模型预测 token+1: '冰'
MTP 生成 3 个草稿: ['城', '<|im_end|>', '\n']

⚠️ 注意：第1个草稿用了主模型的真实 hidden_states，质量较高
但第2、3个草稿用了 MTP 自己的输出拼接——这些 hidden_states 是虚构的!
主模型从未处理过这些 token，拼接出的 hidden states 没有依据，
所以后续草稿几乎等同于随机猜测，而非合理近似。
验证阶段会纠正所有错误，所以最终结果不受影响——
但草稿质量低意味着多草稿策略可能无法带来加速。


## 4. 完整的并行投机解码演示

关键：每轮迭代只有**1次主模型前向**，同时完成验证 + 预测下一个 token。

In [5]:
def speculative_decoding_parallel(input_text, k=3, max_new_tokens=12, verbose=True):
    """并行投机解码：每轮1次主模型前向，验证K个草稿 + 预测下一个token"""
    input_ids = tokenizer.encode(input_text, return_tensors="pt").to(device)
    original_len = len(input_ids[0])
    confirmed = input_ids[0].tolist()
    total_main = 0
    total_mtp = 0

    if verbose:
        print(f"输入: {input_text}")
        print(f"草稿数量 K = {k}")
        print("=" * 70)

    # 初始：1次主模型前向，预测第1个token
    current_ids = torch.tensor([confirmed], device=device)
    with torch.no_grad():
        outputs = model.model(current_ids)
        hidden_states = outputs.last_hidden_state
    total_main += 1

    logits = model.lm_head(hidden_states[:, -1:, :])
    next_id = torch.argmax(F.softmax(logits.float(), dim=-1), dim=-1).item()
    confirmed.append(next_id)

    if verbose:
        print(f"\n[初始] 主模型前向 → 预测 token+1 = '{tokenizer.decode([next_id])}'")

    if next_id == tokenizer.eos_token_id:
        return tokenizer.decode(confirmed), total_main, total_mtp, len(confirmed) - original_len

    # 生成初始K个草稿
    draft_tokens = generate_k_drafts(current_ids, hidden_states, next_id, k=k)
    total_mtp += k

    if verbose:
        draft_texts = [tokenizer.decode([d]) for d in draft_tokens]
        print(f"[初始] MTP 生成 {k} 个草稿: {draft_texts}")

    # 主循环：每轮只有1次主模型前向
    while len(confirmed) - original_len < max_new_tokens:
        # 构建输入：[已确认序列 + K个草稿]
        all_ids = confirmed + draft_tokens
        verify_ids = torch.tensor([all_ids], device=device)

        # ★ 唯一的主模型前向：同时完成验证 + 预测下一个token
        with torch.no_grad():
            outputs = model.model(verify_ids)
            new_hidden = outputs.last_hidden_state
        total_main += 1

        # 验证草稿：逐个检查每个草稿位置的logits
        n_confirmed = len(confirmed)
        accepted = 0
        correction_id = None

        for i, draft_id in enumerate(draft_tokens):
            pos = n_confirmed - 1 + i
            v_logits = model.lm_head(new_hidden[:, pos:pos+1, :])
            verified_id = torch.argmax(F.softmax(v_logits.float(), dim=-1), dim=-1).item()
            if verified_id == draft_id:
                accepted += 1
            else:
                correction_id = verified_id
                break

        # 处理验证结果
        if accepted == k:
            # 全部通过：接受所有草稿 + 末尾额外预测的token
            confirmed.extend(draft_tokens)
            last_logits = model.lm_head(new_hidden[:, -1:, :])
            next_id = torch.argmax(F.softmax(last_logits.float(), dim=-1), dim=-1).item()
            confirmed.append(next_id)

            if verbose:
                print(f"\n  全部 {k} 个草稿通过! +{k+1} tokens (1次主模型前向产出{k+1}个token)")
                print(f"    草稿: {[tokenizer.decode([d]) for d in draft_tokens]}")
                print(f"    额外预测: '{tokenizer.decode([next_id])}'")

            # 因果注意力保证前 n_confirmed+K 个位置的 hidden_states 正确
            mtp_ids = torch.tensor([confirmed[:-1]], device=device)
            mtp_hidden = new_hidden[:, :n_confirmed + k, :]
        else:
            # 部分或全部失败
            confirmed.extend(draft_tokens[:accepted])
            confirmed.append(correction_id)

            if verbose:
                if accepted > 0:
                    print(f"\n  前 {accepted} 个草稿通过，第 {accepted+1} 个失败")
                    print(f"    接受: {[tokenizer.decode([d]) for d in draft_tokens[:accepted]]}")
                else:
                    print(f"\n  第1个草稿就失败")
                print(f"    修正: '{tokenizer.decode([correction_id])}'")

            next_id = correction_id
            mtp_ids = torch.tensor([confirmed[:-1]], device=device)
            mtp_hidden = new_hidden[:, :n_confirmed + accepted, :]

        if verbose:
            print(f"    当前文本: {tokenizer.decode(confirmed)}")

        if next_id == tokenizer.eos_token_id or len(confirmed) - original_len >= max_new_tokens:
            break

        # 生成下一轮草稿
        draft_tokens = generate_k_drafts(mtp_ids, mtp_hidden, next_id, k=k)
        total_mtp += k

    new_tokens = len(confirmed) - original_len
    final_text = tokenizer.decode(confirmed)

    if verbose:
        print(f"\n{'=' * 70}")
        print(f"最终结果: {final_text}")
        print(f"主模型前向: {total_main} 次, MTP前向: {total_mtp} 次")
        print(f"产出 Token: {new_tokens} 个, 加速比: {new_tokens/total_main:.2f}x")

    return final_text, total_main, total_mtp, new_tokens

result_p, main_p, mtp_p, tokens_p = speculative_decoding_parallel("你爱我，我爱你，蜜雪", k=3)

输入: 你爱我，我爱你，蜜雪
草稿数量 K = 3

[初始] 主模型前向 → 预测 token+1 = '冰'
[初始] MTP 生成 3 个草稿: ['城', '<|im_end|>', '\n']

  前 1 个草稿通过，第 2 个失败
    接受: ['城']
    修正: '，'
    当前文本: 你爱我，我爱你，蜜雪冰城，

  前 1 个草稿通过，第 2 个失败
    接受: ['你']
    修正: '爱我'
    当前文本: 你爱我，我爱你，蜜雪冰城，你爱我

  全部 3 个草稿通过! +4 tokens (1次主模型前向产出4个token)
    草稿: ['，', '我爱你', '，']
    额外预测: '蜜'
    当前文本: 你爱我，我爱你，蜜雪冰城，你爱我，我爱你，蜜

  全部 3 个草稿通过! +4 tokens (1次主模型前向产出4个token)
    草稿: ['雪', '冰', '城']
    额外预测: '，'
    当前文本: 你爱我，我爱你，蜜雪冰城，你爱我，我爱你，蜜雪冰城，

最终结果: 你爱我，我爱你，蜜雪冰城，你爱我，我爱你，蜜雪冰城，
主模型前向: 5 次, MTP前向: 12 次
产出 Token: 13 个, 加速比: 2.60x


## 5. K=1 vs K=3：对比演示

用相同的输入对比两种方式，看并行验证的真正优势——以及它的局限：

> 💡 关键问题：K=3 一定比 K=1 快吗？答案取决于**草稿命中率**。如果草稿质量低（如单 MTP 头自回归生成的后续草稿），K=3 可能和 K=1 一样慢，甚至更慢（白跑了多余的 MTP 前向）。

In [6]:
test_inputs = [
    ("你爱我，我爱你，蜜雪", "歌词循环(草稿命中率极高)"),
    ("人工智能的发展历史可以追溯到", "自然文本(草稿命中率中等)"),
]

for input_text, desc in test_inputs:
    print("=" * 70)
    print(f"对比: K=1 vs K=3 — {desc}")
    print(f"输入: {input_text}")
    print("=" * 70)

    result_s, main_s, mtp_s, tokens_s = speculative_decoding_k1(input_text, max_new_tokens=12, verbose=False)
    result_p, main_p, mtp_p, tokens_p = speculative_decoding_parallel(input_text, k=3, max_new_tokens=12, verbose=False)

    print(f"\n{'指标':<20s} {'K=1(单草稿)':<20s} {'K=3(多草稿)':<20s}")
    print("-" * 60)
    print(f"{'产出 Token 数':<20s} {tokens_s:<20d} {tokens_p:<20d}")
    print(f"{'主模型前向次数':<20s} {main_s:<20d} {main_p:<20d}")
    print(f"{'MTP前向次数':<20s} {mtp_s:<20d} {mtp_p:<20d}")
    print(f"{'加速比':<20s} {tokens_s/main_s:<20.2f} {tokens_p/main_p:<20.2f}")

    # 分析草稿命中率
    k1_accept_rate = tokens_s / (main_s + mtp_s) if (main_s + mtp_s) > 0 else 0
    k3_accept_rate = tokens_p / (main_p + mtp_p) if (main_p + mtp_p) > 0 else 0
    print(f"{'整体 token/前向':<20s} {k1_accept_rate:<20.2f} {k3_accept_rate:<20.2f}")
    print()

print("=" * 70)
print("加速比上限:")
print("  K=1: 每轮最多产2个token(1草稿+1预测), 上限 2x")
print("  K=3: 每轮最多产4个token(3草稿+1预测), 上限 4x")
print("  K=5: 每轮最多产6个token(5草稿+1预测), 上限 6x")
print()
print("⚠️ 关键发现: 草稿质量才是决定性因素!")
print()
print("歌词循环案例: K=3 加速比 2.60x > K=1 的 1.86x")
print("  → 高度可预测的文本，草稿命中率高，多草稿策略有效")
print()
print("自然文本案例: K=3 加速比 ≈ K=1")
print("  → 单 MTP 头自回归生成的后续草稿质量太低(虚构 hidden states)")
print("  → 草稿几乎总是第1或第2个就失败，K=3 退化为 K=1")
print("  → 而且还白跑了多余的 MTP 前向，整体效率反而更低")
print()
print("结论: 增大 K 不一定带来加速——草稿质量才是关键。")
print("对于单 MTP 头的模型，K=1 是最稳妥的选择。")
print("只有多头 MTP 或独立草稿模型才能让 K>1 真正发挥优势。")

对比: K=1 vs K=3 — 歌词循环(草稿命中率极高)
输入: 你爱我，我爱你，蜜雪

指标                   K=1(单草稿)             K=3(多草稿)            
------------------------------------------------------------
产出 Token 数           13                   13                  
主模型前向次数              7                    5                   
MTP前向次数              7                    12                  
加速比                  1.86                 2.60                
整体 token/前向          0.93                 0.76                

对比: K=1 vs K=3 — 自然文本(草稿命中率中等)
输入: 人工智能的发展历史可以追溯到

指标                   K=1(单草稿)             K=3(多草稿)            
------------------------------------------------------------
产出 Token 数           13                   13                  
主模型前向次数              7                    7                   
MTP前向次数              7                    18                  
加速比                  1.86                 1.86                
整体 token/前向          0.93                 0.52                

加速比上限:
  K=1: 每轮最多产2个t

## 6. 核心要点总结

### 投机解码的两个阶段

| 阶段 | 执行者 | 次数 | 作用 |
| --- | --- | --- | --- |
| **草稿生成** | MTP 模块 (2.72% 参数量) | K 次 | 快速生成 K 个候选 token |
| **验证+预测** | 主模型 (100% 参数量) | 1 次 | 一次前向验证全部 K 个草稿 + 预测下一个 token |

### 为什么验证K个和验证1个的计算量几乎相同？

主模型的前向传播天然会为序列中的**每个位置**计算 hidden states 和 logits。把 K 个草稿 token 拼在输入序列后面，主模型照常计算，我们只是从输出中多提取了 K 个位置的 logits 做验证——**验证的边际成本极低**，而不是绝对免费。

### 加速比分析

- **K=1（串行）**：每轮最多产 2 个 token → 加速比上限 **2x**
- **K=3（并行）**：每轮最多产 4 个 token → 加速比上限 **4x**
- **K=5（并行）**：每轮最多产 6 个 token → 加速比上限 **6x**

实际加速比取决于草稿命中率。草稿全部错误时反而变慢（白跑了 K 次 MTP），但草稿命中率通常在 60%-80%，实际加速比约 1.5x-3x。

### Qwen3.5 的 MTP 局限性

Qwen3.5-0.8B 只有 1 个 MTP 头（`mtp_num_hidden_layers=1`），所以只能直接预测 token+2。要预测更远的 token，需要自回归地多次运行 MTP，每次用 MTP 自己的输出近似主模型的 hidden_states——**这与训练时"用真实 hidden_states 预测 token+2"存在 gap**，是 MTP 本身设计的局限，导致越远的草稿质量越低。

如果有多个 MTP 头（如 DeepSeek-V3 的 4 个头），每个头专门预测一个偏移量的 token，草稿质量会更高，加速效果也更好。

### 一句话总结

> 投机解码的精髓不是"偷跑1个"，而是"偷跑一串，一次验证"。主模型的1次前向传播是"固定成本"，验证1个草稿和验证K个草稿的计算量几乎相同——增大K才是提升加速比的关键。